# C1 Path-Dependent Exit Management — Phase 1
Fixed global rules, baseline reconciliation first. See `docs/89_c1_path_management_phase1_plan.md`.

## 1. Source / Hash Audit
Set the frozen baseline and M1 root below. Drive mounting and saving are OFF by default.

In [ ]:
from pathlib import Path
import pandas as pd
BASELINE = Path('/content/daily_stop_baseline_trades.csv')
M1_ROOT = Path('/content/m1')
OUT = Path('/content')
DRIVE_SAVE = False
assert BASELINE.exists() and M1_ROOT.exists(), 'Provide the audited inputs before running.'

## 2. R0 Baseline Reconciliation
The command halts on any hash, coverage, or trade mismatch.

In [ ]:
import subprocess, sys
runner = Path('src/research/c1_path_management_phase1.py')
manifest = Path('results/volatility_phase1/volatility_phase1_input_manifest.csv')
subprocess.run([sys.executable,str(runner),'--baseline',str(BASELINE),'--manifest',str(manifest),'--m1-root',str(M1_ROOT),'--out',str(OUT),'--stage','reconcile'],check=True)
display(pd.read_csv(OUT/'c1_path_management_phase1_r0_reconciliation.csv'))
display(pd.read_csv(OUT/'c1_path_management_phase1_m1_audit.csv'))

## 3–10. Path Anatomy, Variants, Periods, CI, Holm, Trade Deltas, Strategy Diagnostics, Gates
Run only after the R0 gate above passes.

In [ ]:
subprocess.run([sys.executable,str(runner),'--baseline',str(BASELINE),'--manifest',str(manifest),'--m1-root',str(M1_ROOT),'--out',str(OUT),'--stage','full'],check=True)
for title,suffix in [('Path Anatomy','path_anatomy_summary'),('Portfolio R0–R3','portfolio_summary'),('Historical / Recent','period_summary'),('Bootstrap and Holm','multiple_comparison'),('Improved vs Harmed','trade_delta_summary'),('Strategy diagnostics','strategy_summary'),('Formal Gate A–G','multiple_comparison')]:
    print('\n'+title)
    frame=pd.read_csv(OUT/f'c1_path_management_phase1_{suffix}.csv')
    display(frame.head(30) if suffix in ('path_anatomy_summary','strategy_summary') else frame)

## 11. Final Verdict · 12. Phase 2 eligibility
Only a `GLOBAL_DYNAMIC_MANAGEMENT_CANDIDATE` may enter separately planned Phase 2. No Dell Demo Phase 5, EA, SET, VPS, or live changes.

In [ ]:
formal=pd.read_csv(OUT/'c1_path_management_phase1_multiple_comparison.csv')
display(formal[['Variant','Label','GateA','GateB','GateC','GateD','GateE','GateF','GateG']])
print('Phase 2:', 'ELIGIBLE FOR SEPARATE PLAN' if (formal.Label=='GLOBAL_DYNAMIC_MANAGEMENT_CANDIDATE').any() else 'DO NOT PROCEED')
if DRIVE_SAVE:
    raise NotImplementedError('Drive save is deliberately OFF; review destination and artifacts before enabling.')